# 06 — Overlays: Vol Targeting + Risk Parity

Two portfolio overlays applied on top of base strategies.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.backtest.runner import run
from src.data.store import get_or_fetch_panel


## VolTarget on TSMOM


In [ ]:
from src.strategies.momentum import TimeSeriesMomentum
from src.strategies.overlays.vol_target import VolTarget

close = get_or_fetch_panel(['SPY', 'QQQ', 'IWM', 'EFA', 'EEM', 'TLT', 'GLD', 'DBC'], '2010-01-01')
base = TimeSeriesMomentum(lookback=252, skip=21)
overlay = VolTarget(base, target_vol=0.10, lookback=20, max_leverage=2.0)
result = run(close, overlay)
print(result.summary())


In [ ]:
pf = result.portfolio
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
pf.value().plot(ax=axes[0]); axes[0].set_title('Equity')
pf.drawdown().plot(ax=axes[1], color='red'); axes[1].set_title('Drawdown')
plt.tight_layout(); plt.show()


In [ ]:
w = overlay.weights(close)
gross = w.abs().sum(axis=1)
fig, ax = plt.subplots(figsize=(11, 3))
gross.plot(ax=ax)
ax.axhline(1.0, color='gray', ls='--', alpha=0.5)
ax.axhline(2.0, color='red', ls='--', alpha=0.5, label='max leverage')
ax.set_title('Gross exposure (sum |weights|)'); ax.legend(); plt.show()


## RiskParity on XSMOM


In [ ]:
from src.strategies.xsmom import CrossSectionalMomentum
from src.strategies.overlays.risk_parity import RiskParity

base = CrossSectionalMomentum(lookback=252, skip=21, top_n=3)
overlay = RiskParity(base, lookback=60)
result = run(close, overlay)
print(result.summary())


In [ ]:
pf = result.portfolio
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
pf.value().plot(ax=axes[0]); axes[0].set_title('Equity')
pf.drawdown().plot(ax=axes[1], color='red'); axes[1].set_title('Drawdown')
plt.tight_layout(); plt.show()


In [ ]:
w = overlay.weights(close)
fig, ax = plt.subplots(figsize=(11, 4))
w.iloc[-500:].plot.area(ax=ax, stacked=True)
ax.set_ylim(0, 1.05); ax.set_title('Risk-parity weights (last 500 bars)')
ax.legend(loc='upper left', bbox_to_anchor=(1, 1)); plt.show()
